In [ ]:
# Общий просмотр всех 11 таблиц lakehouse
# Principal: ivan | Role: data_engineer
# Credentials: IVAN_CLIENT_ID / IVAN_CLIENT_SECRET

In [ ]:
import os
from pyspark.sql import SparkSession

client_id = os.environ["IVAN_CLIENT_ID"]
client_secret = os.environ["IVAN_CLIENT_SECRET"]
credential = f"{client_id}:{client_secret}"

spark = SparkSession.builder \
    .appName("lakehouse-select") \
    .config("spark.sql.catalog.lakehouse.credential", credential) \
    .getOrCreate()

spark

In [ ]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [ ]:
print("Bronze raw_categories: справочник категорий")
spark.sql("""
    SELECT
        id,
        name,
        description
    FROM lakehouse.bronze.raw_categories
    LIMIT 3
""").show(truncate=False)

print("Bronze raw_products: сырой каталог товаров")
spark.sql("""
    SELECT
        id,
        category_id,
        name,
        price,
        is_active
    FROM lakehouse.bronze.raw_products
    LIMIT 3
""").show(truncate=False)

print("Bronze raw_customers: покупатели с дублями, created_at как STRING")
spark.sql("""
    SELECT
        id,
        name,
        email,
        city,
        created_at
    FROM lakehouse.bronze.raw_customers
    LIMIT 3
""").show(truncate=False)

print("Bronze raw_orders: заказы с невалидными статусами DONE/shipped")
spark.sql("""
    SELECT
        id,
        customer_id,
        status,
        total_amount,
        order_date
    FROM lakehouse.bronze.raw_orders
    LIMIT 3
""").show(truncate=False)

print("Bronze raw_order_items: позиции заказов")
spark.sql("""
    SELECT
        id,
        order_id,
        product_id,
        quantity,
        unit_price
    FROM lakehouse.bronze.raw_order_items
    LIMIT 3
""").show(truncate=False)

In [ ]:
print("Silver customers: после дедупликации")
spark.sql("""
    SELECT
        id,
        name,
        email,
        city,
        created_at
    FROM lakehouse.silver.customers
    LIMIT 3
""").show(truncate=False)

print("Silver products: только активные с категорией")
spark.sql("""
    SELECT
        id,
        category_id,
        category_name,
        name,
        price
    FROM lakehouse.silver.products
    LIMIT 3
""").show(truncate=False)

print("Silver orders: только валидные статусы")
spark.sql("""
    SELECT
        id,
        customer_id,
        status,
        total_amount,
        order_date
    FROM lakehouse.silver.orders
    LIMIT 3
""").show(truncate=False)

print("Silver order_items: с вычисленным line_total")
spark.sql("""
    SELECT
        id,
        order_id,
        product_id,
        quantity,
        unit_price,
        line_total
    FROM lakehouse.silver.order_items
    LIMIT 3
""").show(truncate=False)

In [ ]:
print("Gold mart_sales_by_category: выручка по категориям и месяцам")
spark.sql("""
    SELECT
        category_name,
        month,
        total_revenue,
        order_count,
        avg_check
    FROM lakehouse.gold.mart_sales_by_category
    LIMIT 3
""").show(truncate=False)

print("Gold mart_top_customers: RFM-сегментация покупателей")
spark.sql("""
    SELECT
        customer_id,
        name,
        recency_days,
        frequency,
        monetary,
        segment
    FROM lakehouse.gold.mart_top_customers
    LIMIT 3
""").show(truncate=False)